In [1]:
import matplotlib
%matplotlib inline
from pylab import *
import caesar
import h5py, os, sys
import yt, pickle
from scipy.optimize import curve_fit
from scipy.stats import binned_statistic as bst
from scipy.stats import binned_statistic_2d as bst2
from scipy.signal import savgol_filter as sgf

matplotlib.rcParams['xtick.labelsize'] = 18
matplotlib.rcParams['ytick.labelsize'] = 18

safecolor={
    'silver':"#c0c0c0",
    'gray':"#808080",
    'black':"#000000",
    'red':"#ff0000",
    'maroon':"#800000",
    'yellow':"#ffff00",
    'olive':"#707030",
    'lime':"#00ee00",
    'green':"#009000",
    'aqua':"#00eeee",
    'teal':"#00a0a0",
    'blue':"#0000ff",
    'nave':"#000080",
    'fuchsia':"#ff00ff",
    'purple':"#900090"
}

import halo_profile as hp

import glob

# we will use the AHF profile to calculate R500, M500, M*500 and Mgas500 and concentration

In [31]:
def get_ahf_500c(mod, rg, sn):
    ahfp='/home2/weiguang/The300/data/catalogues/AHF/'+mod+'/'
    snap='snap_%03d'%sn
    if mod == 'GIZMO_7k_R200c':
        cn = 'NewMDCLUSTER_%03d'%rg
        filen=ahfp+cn+'/GIZMO_7k'+'-%s.%s.z*.AHF_halos' %(cn,snap)
    else:
        cn = 'NewMDCLUSTER_%04d'%rg
        filen=ahfp+cn+'/'+mod+'-%s.%s.z*.AHF_halos' %(cn,snap)

    fahf = glob.glob(filen)
    if len(fahf)!=1:
        print('File not found!', filen, fahf)
        return None
    # now open files for reading and analysing
    ahf_h = np.loadtxt(fahf[0],usecols=(0,1,3,11,36,37,44,64)) #id, host, M, r, nbin, fhires mg m*
    if mod == 'GIZMO_7k_R200c':
        ahf_p = np.loadtxt(fahf[0][:-5]+'profiles', usecols=(0,2,3,4,24,25)) #r, M, dens, Mgas, M*
    else:
        ahf_p = np.loadtxt(fahf[0][:-5]+'profiles.gz', usecols=(0,2,3,4,24,25)) #r, M, dens, Mgas, M*

    # print(ahf_h.shape,ahf_p.shape)
    prof=np.zeros((ahf_h.shape[0],10),dtype=np.float64) #ID, M200, R200, Mgas200, M*200; M500, R500, M*500 and Mgas500 and concentration !only host halos!
    count=0
    cp=0
    ch=0
    while ch<ahf_h.shape[0]:
        if (ahf_h[ch,1] < 1) & (ahf_h[ch,5]>0.985):  #host halo
            prof[count,0:5]=[ahf_h[ch,0],ahf_h[ch,2],ahf_h[ch,3],ahf_h[ch,6],ahf_h[ch,7]]
            nbin=np.int32(ahf_h[ch,4])
            
            tmpr=np.log10(np.abs(ahf_p[cp:cp+nbin, 0]))
            tmpd=ahf_p[cp:cp+nbin, 2]
            intr=np.interp(500, tmpd, tmpr)
            prof[count, 6]=10**intr
            prof[count, 5]=10**np.interp(intr, tmpr, np.log10(ahf_p[cp:cp+nbin, 1]))
            prof[count, 7]=10**np.interp(intr, tmpr, np.log10(ahf_p[cp:cp+nbin, 4]))
            prof[count, 8]=10**np.interp(intr, tmpr, np.log10(ahf_p[cp:cp+nbin, 5]))

            #profile fitting
            idr=(10**tmpr>=0.05*ahf_h[ch,3]) & (ahf_p[cp:cp+nbin, 0]>0)
            if len(tmpr[idr])>9:
                prof[count, 9] = ahf_h[ch,3]/hp.profit(10**tmpr[idr], ahf_p[cp:cp+nbin, 3][idr], maxfev=5000)[1]
            else:
                prof[count, 9] = -1
            # print(cp,nbin,ahf_p[cp:cp+nbin, 0])
            count+=1
            ch+=1
            cp+=nbin
            
        else:

            cp+=np.int32(ahf_h[ch,4])
            ch+=1
            # print('subhalo',ch,cp)
            
    return prof[:count]

In [9]:
denst=get_ahf_500c('GIZMO_7k_R200c', 1, 128)

(98844, 7) (711389, 6)


/tmp/ipykernel_1532301/571562813.py:37: RuntimeWarning: divide by zero encountered in log10
  prof[count, 7]=10**np.interp(intr, tmpr, np.log10(ahf_p[cp:cp+nbin, 4]))
/tmp/ipykernel_1532301/571562813.py:38: RuntimeWarning: divide by zero encountered in log10
  prof[count, 8]=10**np.interp(intr, tmpr, np.log10(ahf_p[cp:cp+nbin, 5]))


In [30]:
fh5w = h5py.File('R500_concentration_information.hdf5', "w")
fh5w.create_dataset('README', data = '#ID, M200, R200, Mgas200, M*200; M500, R500, M*500 and Mgas500 and concentration !only host halos!')
for i in np.arange(1,325):
    print(i)
    tpd=get_ahf_500c('GIZMO_7k_R200c', 1, 128)
    if tpd is not None:
        fh5w.create_dataset('%3d'%i, data=tpd)
fh5w.close()

1


/tmp/ipykernel_1532301/1747438289.py:37: RuntimeWarning: divide by zero encountered in log10
  prof[count, 7]=10**np.interp(intr, tmpr, np.log10(ahf_p[cp:cp+nbin, 4]))
/tmp/ipykernel_1532301/1747438289.py:38: RuntimeWarning: divide by zero encountered in log10
  prof[count, 8]=10**np.interp(intr, tmpr, np.log10(ahf_p[cp:cp+nbin, 5]))
/home2/weiguang/.local/pylib/halo_profile.py:34: RuntimeWarning: invalid value encountered in log10
  return float64(a + 3.0*log10(b) - x - 2.0*log10(b + 10.**x))


2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
27

In [ ]:
fh5w = h5py.File('R500_concentration_information_128.hdf5', "w")
st= '#ID[0], M200[1], R200[2], Mgas200[3], M*200[4]; M500[5], R500[6], Mgas500[7], M*500[8] and concentration[9] !only host halos!'
dt = h5py.special_dtype(vlen=str)
dset = fh5w.create_dataset('README', (1,), dtype=dt)
dset[0] = st
# fh5w.create_dataset('README',  (len(st),1),'S%d'%len(st), st)
for i in np.arange(1,325):
    print(i)
    tpd=get_ahf_500c('GIZMO_7k_R200c', i, 128)
    if tpd is not None:
        fh5w.create_dataset('NewMDCLUSTER_%03d'%i, data=tpd)
fh5w.close()

1


/tmp/ipykernel_1532301/1649302889.py:37: RuntimeWarning: divide by zero encountered in log10
  prof[count, 7]=10**np.interp(intr, tmpr, np.log10(ahf_p[cp:cp+nbin, 4]))
/tmp/ipykernel_1532301/1649302889.py:38: RuntimeWarning: divide by zero encountered in log10
  prof[count, 8]=10**np.interp(intr, tmpr, np.log10(ahf_p[cp:cp+nbin, 5]))


2
3
4
5
6
7
8
9
10
11
12


In [43]:
fh5w.close()

In [39]:
len(st)

125